In [ ]:
library(tidyverse)
library(bigrquery)

# The below queries represent dataset AllOfUsControlledTierDatasetv9 in cohort hhtepi at 20260830_030241.

# If EXPORT_BUCKET is not set in your environment, uncomment and update the line below
# with your actual Cloud Storage bucket name from the Resources tab in your workspace.
Sys.setenv(EXPORT_BUCKET = "gs://hht-epi-bucket-wb-lucent-greens-2488")


# Domain person
person_sql = r"(

    SELECT
        st.birth_datetime,
        st.ethnicity_concept_id,
        dt0.concept_name AS T_DISP_ethnicity,
        st.gender_concept_id,
        dt1.concept_name AS T_DISP_gender,
        st.person_id,
        st.race_concept_id,
        dt2.concept_name AS T_DISP_race,
        st.self_reported_category_concept_id,
        dt3.concept_name AS T_DISP_self_reported_category,
        st.sex_at_birth_concept_id,
        dt4.concept_name AS T_DISP_sex_at_birth 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.person AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.ethnicity_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.gender_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.race_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt3 
            ON dt3.concept_id = st.self_reported_category_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt4 
            ON dt4.concept_id = st.sex_at_birth_concept_id 
    WHERE
        st.person_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_person 
        WHERE
            (id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
            WHERE
                condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor = 313504 
                UNION
                ALL SELECT
                    313504))) 
                OR ((id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ENG')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ACVRL1')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN ('18-51067120-TAGAC-T', '18-51058355-G-A', '18-51048839-C-T', '18-51076662-C-T', '18-51076653-C-T', '18-51078306-A-G', '18-51078294-C-T', '10-47325545-T-C')))))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
person_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports",
  "person",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "person",
  "person_*.csv")
message(str_glue('The data will be written to {person_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", person_sql, billing = "wb-lucent-greens-2488"),
  person_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {person_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

person_df <- read_bq_export_from_workspace_bucket(person_path)
dim(person_df)
head(person_df, 5)


# Domain conditionOccurrence
conditionOccurrence_sql = r"(

    SELECT
        st.condition_concept_id,
        st.condition_end_datetime,
        st.condition_source_concept_id,
        st.condition_source_value,
        st.condition_start_datetime,
        st.condition_status_concept_id,
        dt0.concept_name AS T_DISP_condition_status_concept_name,
        st.condition_status_source_value,
        st.condition_type_concept_id,
        dt1.concept_name AS T_DISP_condition_type_concept_name,
        st.person_id,
        dt2.concept_code AS T_DISP_source_concept_code,
        dt2.concept_name AS T_DISP_source_concept_name,
        dt2.vocabulary_id AS T_DISP_source_vocabulary,
        dt3.concept_code AS T_DISP_standard_concept_code,
        dt3.concept_name AS T_DISP_standard_concept_name,
        dt3.vocabulary_id AS T_DISP_standard_vocabulary,
        st.stop_reason,
        st.visit_occurrence_id 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.condition_occurrence AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.condition_status_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.condition_type_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.condition_source_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt3 
            ON dt3.concept_id = st.condition_concept_id 
    WHERE
        st.condition_occurrence_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_conditionOccurrence_person_id 
        WHERE
            (((person_id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
            WHERE
                condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor = 313504 
                UNION
                ALL SELECT
                    313504))) 
                OR ((person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ENG')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ACVRL1')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN ('18-51067120-TAGAC-T', '18-51058355-G-A', '18-51048839-C-T', '18-51076662-C-T', '18-51076653-C-T', '18-51078306-A-G', '18-51078294-C-T', '10-47325545-T-C')))))) 
                AND (condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor IN (4051478, 4096682, 37170305) 
                UNION
                ALL SELECT
                    4051478 
                UNION
                ALL SELECT
                    4096682 
                UNION
                ALL SELECT
                    37170305))) 
                OR (((person_id IN (SELECT
                    person_id AS primary_id 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
                WHERE
                    condition_concept_id IN (SELECT
                        descendant 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                    WHERE
                        ancestor = 313504 
                    UNION
                    ALL SELECT
                        313504))) 
                    OR ((person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ENG')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                    OR (person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ACVRL1')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                    OR (person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN ('18-51067120-TAGAC-T', '18-51058355-G-A', '18-51048839-C-T', '18-51076662-C-T', '18-51076653-C-T', '18-51078306-A-G', '18-51078294-C-T', '10-47325545-T-C')))))) 
                    AND (condition_source_concept_id IN (SELECT
                        descendant 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_icd10cmConcept_default 
                    WHERE
                        ancestor = 45548892 
                    UNION
                    ALL SELECT
                        45548892))) 
                    OR (((person_id IN (SELECT
                        person_id AS primary_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
                    WHERE
                        condition_concept_id IN (SELECT
                            descendant 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                        WHERE
                            ancestor = 313504 
                        UNION
                        ALL SELECT
                            313504))) 
                        OR ((person_id IN (SELECT
                            flattened_person_id 
                        FROM
                            (SELECT
                                DISTINCT vid, flattened_person_id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                            CROSS JOIN
                                UNNEST(ARRAY(SELECT
                                    element 
                                FROM
                                    UNNEST(person_ids.list))) AS flattened_person_id 
                            JOIN
                                `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                    ON sp.person_id = flattened_person_id 
                            WHERE
                                vid IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                                WHERE
                                    (id IN (SELECT
                                        id 
                                    FROM
                                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                    WHERE
                                        gene = 'ENG')) 
                                    AND (EXISTS (SELECT
                                        * 
                                    FROM
                                        UNNEST(clinvar_significance) AS flattened 
                                    WHERE
                                        flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                        OR (person_id IN (SELECT
                            flattened_person_id 
                        FROM
                            (SELECT
                                DISTINCT vid, flattened_person_id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                            CROSS JOIN
                                UNNEST(ARRAY(SELECT
                                    element 
                                FROM
                                    UNNEST(person_ids.list))) AS flattened_person_id 
                            JOIN
                                `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                    ON sp.person_id = flattened_person_id 
                            WHERE
                                vid IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                                WHERE
                                    (id IN (SELECT
                                        id 
                                    FROM
                                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                    WHERE
                                        gene = 'ACVRL1')) 
                                    AND (EXISTS (SELECT
                                        * 
                                    FROM
                                        UNNEST(clinvar_significance) AS flattened 
                                    WHERE
                                        flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                        OR (person_id IN (SELECT
                            flattened_person_id 
                        FROM
                            (SELECT
                                DISTINCT vid, flattened_person_id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                            CROSS JOIN
                                UNNEST(ARRAY(SELECT
                                    element 
                                FROM
                                    UNNEST(person_ids.list))) AS flattened_person_id 
                            JOIN
                                `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                    ON sp.person_id = flattened_person_id 
                            WHERE
                                vid IN ('18-51067120-TAGAC-T', '18-51058355-G-A', '18-51048839-C-T', '18-51076662-C-T', '18-51076653-C-T', '18-51078306-A-G', '18-51078294-C-T', '10-47325545-T-C')))))) 
                        AND (condition_concept_id IN (SELECT
                            descendant 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                        WHERE
                            ancestor IN (192671, 432875, 436659, 439777, 439847, 4049224, 4051478, 4096682, 4119302, 4121803, 4146209, 37162759, 37170305) 
                        UNION
                        ALL SELECT
                            192671 
                        UNION
                        ALL SELECT
                            432875 
                        UNION
                        ALL SELECT
                            436659 
                        UNION
                        ALL SELECT
                            439777 
                        UNION
                        ALL SELECT
                            439847 
                        UNION
                        ALL SELECT
                            4049224 
                        UNION
                        ALL SELECT
                            4051478 
                        UNION
                        ALL SELECT
                            4096682 
                        UNION
                        ALL SELECT
                            4119302 
                        UNION
                        ALL SELECT
                            4121803 
                        UNION
                        ALL SELECT
                            4146209 
                        UNION
                        ALL SELECT
                            37162759 
                        UNION
                        ALL SELECT
                            37170305))) 
                        OR (((person_id IN (SELECT
                            person_id AS primary_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
                        WHERE
                            condition_concept_id IN (SELECT
                                descendant 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                            WHERE
                                ancestor = 313504 
                            UNION
                            ALL SELECT
                                313504))) 
                            OR ((person_id IN (SELECT
                                flattened_person_id 
                            FROM
                                (SELECT
                                    DISTINCT vid, flattened_person_id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                                CROSS JOIN
                                    UNNEST(ARRAY(SELECT
                                        element 
                                    FROM
                                        UNNEST(person_ids.list))) AS flattened_person_id 
                                JOIN
                                    `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                        ON sp.person_id = flattened_person_id 
                                WHERE
                                    vid IN (SELECT
                                        id 
                                    FROM
                                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                                    WHERE
                                        (id IN (SELECT
                                            id 
                                        FROM
                                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                        WHERE
                                            gene = 'ENG')) 
                                        AND (EXISTS (SELECT
                                            * 
                                        FROM
                                            UNNEST(clinvar_significance) AS flattened 
                                        WHERE
                                            flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                            OR (person_id IN (SELECT
                                flattened_person_id 
                            FROM
                                (SELECT
                                    DISTINCT vid, flattened_person_id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                                CROSS JOIN
                                    UNNEST(ARRAY(SELECT
                                        element 
                                    FROM
                                        UNNEST(person_ids.list))) AS flattened_person_id 
                                JOIN
                                    `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                        ON sp.person_id = flattened_person_id 
                                WHERE
                                    vid IN (SELECT
                                        id 
                                    FROM
                                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                                    WHERE
                                        (id IN (SELECT
                                            id 
                                        FROM
                                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                        WHERE
                                            gene = 'ACVRL1')) 
                                        AND (EXISTS (SELECT
                                            * 
                                        FROM
                                            UNNEST(clinvar_significance) AS flattened 
                                        WHERE
                                            flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                            OR (person_id IN (SELECT
                                flattened_person_id 
                            FROM
                                (SELECT
                                    DISTINCT vid, flattened_person_id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                                CROSS JOIN
                                    UNNEST(ARRAY(SELECT
                                        element 
                                    FROM
                                        UNNEST(person_ids.list))) AS flattened_person_id 
                                JOIN
                                    `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                        ON sp.person_id = flattened_person_id 
                                WHERE
                                    vid IN ('18-51067120-TAGAC-T', '18-51058355-G-A', '18-51048839-C-T', '18-51076662-C-T', '18-51076653-C-T', '18-51078306-A-G', '18-51078294-C-T', '10-47325545-T-C')))))) 
                            AND (condition_concept_id IN (SELECT
                                descendant 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                            WHERE
                                ancestor = 313504 
                            UNION
                            ALL SELECT
                                313504)))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
conditionOccurrence_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports",
  "conditionOccurrence",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "conditionOccurrence",
  "conditionOccurrence_*.csv")
message(str_glue('The data will be written to {conditionOccurrence_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", conditionOccurrence_sql, billing = "wb-lucent-greens-2488"),
  conditionOccurrence_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {conditionOccurrence_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

conditionOccurrence_df <- read_bq_export_from_workspace_bucket(conditionOccurrence_path)
dim(conditionOccurrence_df)
head(conditionOccurrence_df, 5)


# Domain measurementOccurrence
measurementOccurrence_sql = r"(

    SELECT
        st.measurement_concept_id,
        st.measurement_datetime,
        st.measurement_source_concept_id,
        st.measurement_source_value,
        st.measurement_type_concept_id,
        dt0.concept_name AS T_DISP_measurement_type_concept_name,
        st.operator_concept_id,
        dt1.concept_name AS T_DISP_operator_concept_name,
        st.person_id,
        st.range_high,
        st.range_low,
        dt2.concept_code AS T_DISP_source_concept_code,
        dt2.concept_name AS T_DISP_source_concept_name,
        dt2.vocabulary_id AS T_DISP_source_vocabulary,
        dt3.concept_code AS T_DISP_standard_concept_code,
        dt3.concept_name AS T_DISP_standard_concept_name,
        dt3.vocabulary_id AS T_DISP_standard_vocabulary,
        st.unit_concept_id,
        dt4.concept_name AS T_DISP_unit_concept_name,
        st.unit_source_value,
        st.value_as_concept_id,
        dt5.concept_name AS T_DISP_value_as_concept_name,
        st.value_as_number,
        st.value_source_value,
        st.visit_occurrence_id 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.measurement AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.measurement_type_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.operator_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.measurement_source_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt3 
            ON dt3.concept_id = st.measurement_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt4 
            ON dt4.concept_id = st.unit_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt5 
            ON dt5.concept_id = st.value_as_concept_id 
    WHERE
        st.measurement_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_measurementOccurrence_person_id 
        WHERE
            (((person_id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
            WHERE
                condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor = 313504 
                UNION
                ALL SELECT
                    313504))) 
                OR ((person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ENG')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ACVRL1')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN ('18-51067120-TAGAC-T', '18-51058355-G-A', '18-51048839-C-T', '18-51076662-C-T', '18-51076653-C-T', '18-51078306-A-G', '18-51078294-C-T', '10-47325545-T-C')))))) 
                AND (measurement_source_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_icd10cmConcept_default 
                WHERE
                    ancestor = 45548892 
                UNION
                ALL SELECT
                    45548892))) 
                OR (((person_id IN (SELECT
                    person_id AS primary_id 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
                WHERE
                    condition_concept_id IN (SELECT
                        descendant 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                    WHERE
                        ancestor = 313504 
                    UNION
                    ALL SELECT
                        313504))) 
                    OR ((person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ENG')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                    OR (person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ACVRL1')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                    OR (person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN ('18-51067120-TAGAC-T', '18-51058355-G-A', '18-51048839-C-T', '18-51076662-C-T', '18-51076653-C-T', '18-51078306-A-G', '18-51078294-C-T', '10-47325545-T-C')))))) 
                    AND (id IN (SELECT
                        entity_A_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_RIDS_cpt4Person_measurementOccurrence_cpt4Concept 
                    WHERE
                        entity_B_id IN (SELECT
                            descendant 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_cpt4Concept_default 
                        WHERE
                            ancestor IN (7146, 7147, 7148, 7149, 8757, 8761, 8762, 11925, 11928, 12568, 12569, 12570) 
                        UNION
                        ALL SELECT
                            7146 
                        UNION
                        ALL SELECT
                            7147 
                        UNION
                        ALL SELECT
                            7148 
                        UNION
                        ALL SELECT
                            7149 
                        UNION
                        ALL SELECT
                            8757 
                        UNION
                        ALL SELECT
                            8761 
                        UNION
                        ALL SELECT
                            8762 
                        UNION
                        ALL SELECT
                            11925 
                        UNION
                        ALL SELECT
                            11928 
                        UNION
                        ALL SELECT
                            12568 
                        UNION
                        ALL SELECT
                            12569 
                        UNION
                        ALL SELECT
                            12570))))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
measurementOccurrence_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports",
  "measurementOccurrence",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "measurementOccurrence",
  "measurementOccurrence_*.csv")
message(str_glue('The data will be written to {measurementOccurrence_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", measurementOccurrence_sql, billing = "wb-lucent-greens-2488"),
  measurementOccurrence_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {measurementOccurrence_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

measurementOccurrence_df <- read_bq_export_from_workspace_bucket(measurementOccurrence_path)
dim(measurementOccurrence_df)
head(measurementOccurrence_df, 5)


# Domain observationOccurrence
observationOccurrence_sql = r"(

    SELECT
        st.observation_concept_id,
        st.observation_datetime,
        st.observation_source_concept_id,
        st.observation_source_value,
        st.observation_type_concept_id,
        dt0.concept_name AS T_DISP_observation_type_concept_name,
        st.person_id,
        st.qualifier_concept_id,
        dt1.concept_name AS T_DISP_qualifier_concept_name,
        st.qualifier_source_value,
        st.questionnaire_response_id,
        dt2.concept_code AS T_DISP_source_concept_code,
        dt2.concept_name AS T_DISP_source_concept_name,
        dt2.vocabulary_id AS T_DISP_source_vocabulary,
        dt3.concept_code AS T_DISP_standard_concept_code,
        dt3.concept_name AS T_DISP_standard_concept_name,
        dt3.vocabulary_id AS T_DISP_standard_vocabulary,
        st.unit_concept_id,
        dt4.concept_name AS T_DISP_unit_concept_name,
        st.unit_source_value,
        st.value_as_concept_id,
        dt5.concept_name AS T_DISP_value_as_concept_name,
        st.value_as_number,
        st.value_as_string,
        st.value_source_concept_id,
        st.value_source_value,
        st.visit_occurrence_id 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.observation AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.observation_type_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.qualifier_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.observation_source_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt3 
            ON dt3.concept_id = st.observation_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt4 
            ON dt4.concept_id = st.unit_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt5 
            ON dt5.concept_id = st.value_as_concept_id 
    WHERE
        st.observation_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_observationOccurrence_person_id 
        WHERE
            (((person_id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
            WHERE
                condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor = 313504 
                UNION
                ALL SELECT
                    313504))) 
                OR ((person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ENG')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ACVRL1')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN ('18-51067120-TAGAC-T', '18-51058355-G-A', '18-51048839-C-T', '18-51076662-C-T', '18-51076653-C-T', '18-51078306-A-G', '18-51078294-C-T', '10-47325545-T-C')))))) 
                AND (observation_source_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_icd10cmConcept_default 
                WHERE
                    ancestor = 45548892 
                UNION
                ALL SELECT
                    45548892))) 
                OR (((person_id IN (SELECT
                    person_id AS primary_id 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
                WHERE
                    condition_concept_id IN (SELECT
                        descendant 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                    WHERE
                        ancestor = 313504 
                    UNION
                    ALL SELECT
                        313504))) 
                    OR ((person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ENG')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                    OR (person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ACVRL1')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                    OR (person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN ('18-51067120-TAGAC-T', '18-51058355-G-A', '18-51048839-C-T', '18-51076662-C-T', '18-51076653-C-T', '18-51078306-A-G', '18-51078294-C-T', '10-47325545-T-C')))))) 
                    AND (id IN (SELECT
                        entity_A_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_RIDS_cpt4Person_observationOccurrence_cpt4Concept 
                    WHERE
                        entity_B_id IN (SELECT
                            descendant 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_cpt4Concept_default 
                        WHERE
                            ancestor IN (7146, 7147, 7148, 7149, 8757, 8761, 8762, 11925, 11928, 12568, 12569, 12570) 
                        UNION
                        ALL SELECT
                            7146 
                        UNION
                        ALL SELECT
                            7147 
                        UNION
                        ALL SELECT
                            7148 
                        UNION
                        ALL SELECT
                            7149 
                        UNION
                        ALL SELECT
                            8757 
                        UNION
                        ALL SELECT
                            8761 
                        UNION
                        ALL SELECT
                            8762 
                        UNION
                        ALL SELECT
                            11925 
                        UNION
                        ALL SELECT
                            11928 
                        UNION
                        ALL SELECT
                            12568 
                        UNION
                        ALL SELECT
                            12569 
                        UNION
                        ALL SELECT
                            12570))))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
observationOccurrence_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports",
  "observationOccurrence",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "observationOccurrence",
  "observationOccurrence_*.csv")
message(str_glue('The data will be written to {observationOccurrence_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", observationOccurrence_sql, billing = "wb-lucent-greens-2488"),
  observationOccurrence_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {observationOccurrence_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

observationOccurrence_df <- read_bq_export_from_workspace_bucket(observationOccurrence_path)
dim(observationOccurrence_df)
head(observationOccurrence_df, 5)


# Domain procedureOccurrence
procedureOccurrence_sql = r"(

    SELECT
        st.modifier_concept_id,
        dt0.concept_name AS T_DISP_modifier_concept_name,
        st.modifier_source_value,
        st.person_id,
        st.procedure_concept_id,
        st.procedure_datetime,
        st.procedure_source_concept_id,
        st.procedure_source_value,
        st.procedure_type_concept_id,
        dt1.concept_name AS T_DISP_procedure_type_concept_name,
        st.quantity,
        dt2.concept_code AS T_DISP_source_concept_code,
        dt2.concept_name AS T_DISP_source_concept_name,
        dt2.vocabulary_id AS T_DISP_source_vocabulary,
        dt3.concept_code AS T_DISP_standard_concept_code,
        dt3.concept_name AS T_DISP_standard_concept_name,
        dt3.vocabulary_id AS T_DISP_standard_vocabulary,
        st.visit_occurrence_id 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.procedure_occurrence AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.modifier_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.procedure_type_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.procedure_source_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt3 
            ON dt3.concept_id = st.procedure_concept_id 
    WHERE
        st.procedure_occurrence_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_procedureOccurrence_person_id 
        WHERE
            (((person_id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
            WHERE
                condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor = 313504 
                UNION
                ALL SELECT
                    313504))) 
                OR ((person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ENG')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ACVRL1')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN ('18-51067120-TAGAC-T', '18-51058355-G-A', '18-51048839-C-T', '18-51076662-C-T', '18-51076653-C-T', '18-51078306-A-G', '18-51078294-C-T', '10-47325545-T-C')))))) 
                AND (procedure_source_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_icd10cmConcept_default 
                WHERE
                    ancestor = 45548892 
                UNION
                ALL SELECT
                    45548892))) 
                OR (((person_id IN (SELECT
                    person_id AS primary_id 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
                WHERE
                    condition_concept_id IN (SELECT
                        descendant 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                    WHERE
                        ancestor = 313504 
                    UNION
                    ALL SELECT
                        313504))) 
                    OR ((person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ENG')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                    OR (person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ACVRL1')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                    OR (person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN ('18-51067120-TAGAC-T', '18-51058355-G-A', '18-51048839-C-T', '18-51076662-C-T', '18-51076653-C-T', '18-51078306-A-G', '18-51078294-C-T', '10-47325545-T-C')))))) 
                    AND (procedure_concept_id IN (SELECT
                        descendant 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_procedureConcept_default 
                    WHERE
                        ancestor IN (4022173, 4046732, 4122427, 4125928, 4144919, 4218549, 4220838, 4249893, 4262407, 4294382, 37156556, 40485932) 
                    UNION
                    ALL SELECT
                        4022173 
                    UNION
                    ALL SELECT
                        4046732 
                    UNION
                    ALL SELECT
                        4122427 
                    UNION
                    ALL SELECT
                        4125928 
                    UNION
                    ALL SELECT
                        4144919 
                    UNION
                    ALL SELECT
                        4218549 
                    UNION
                    ALL SELECT
                        4220838 
                    UNION
                    ALL SELECT
                        4249893 
                    UNION
                    ALL SELECT
                        4262407 
                    UNION
                    ALL SELECT
                        4294382 
                    UNION
                    ALL SELECT
                        37156556 
                    UNION
                    ALL SELECT
                        40485932))) 
                    OR (((person_id IN (SELECT
                        person_id AS primary_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
                    WHERE
                        condition_concept_id IN (SELECT
                            descendant 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                        WHERE
                            ancestor = 313504 
                        UNION
                        ALL SELECT
                            313504))) 
                        OR ((person_id IN (SELECT
                            flattened_person_id 
                        FROM
                            (SELECT
                                DISTINCT vid, flattened_person_id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                            CROSS JOIN
                                UNNEST(ARRAY(SELECT
                                    element 
                                FROM
                                    UNNEST(person_ids.list))) AS flattened_person_id 
                            JOIN
                                `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                    ON sp.person_id = flattened_person_id 
                            WHERE
                                vid IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                                WHERE
                                    (id IN (SELECT
                                        id 
                                    FROM
                                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                    WHERE
                                        gene = 'ENG')) 
                                    AND (EXISTS (SELECT
                                        * 
                                    FROM
                                        UNNEST(clinvar_significance) AS flattened 
                                    WHERE
                                        flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                        OR (person_id IN (SELECT
                            flattened_person_id 
                        FROM
                            (SELECT
                                DISTINCT vid, flattened_person_id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                            CROSS JOIN
                                UNNEST(ARRAY(SELECT
                                    element 
                                FROM
                                    UNNEST(person_ids.list))) AS flattened_person_id 
                            JOIN
                                `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                    ON sp.person_id = flattened_person_id 
                            WHERE
                                vid IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                                WHERE
                                    (id IN (SELECT
                                        id 
                                    FROM
                                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                    WHERE
                                        gene = 'ACVRL1')) 
                                    AND (EXISTS (SELECT
                                        * 
                                    FROM
                                        UNNEST(clinvar_significance) AS flattened 
                                    WHERE
                                        flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                        OR (person_id IN (SELECT
                            flattened_person_id 
                        FROM
                            (SELECT
                                DISTINCT vid, flattened_person_id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                            CROSS JOIN
                                UNNEST(ARRAY(SELECT
                                    element 
                                FROM
                                    UNNEST(person_ids.list))) AS flattened_person_id 
                            JOIN
                                `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                    ON sp.person_id = flattened_person_id 
                            WHERE
                                vid IN ('18-51067120-TAGAC-T', '18-51058355-G-A', '18-51048839-C-T', '18-51076662-C-T', '18-51076653-C-T', '18-51078306-A-G', '18-51078294-C-T', '10-47325545-T-C')))))) 
                        AND (id IN (SELECT
                            entity_A_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_RIDS_cpt4Person_procedureOccurrence_cpt4Concept 
                        WHERE
                            entity_B_id IN (SELECT
                                descendant 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_cpt4Concept_default 
                            WHERE
                                ancestor IN (7146, 7147, 7148, 7149, 8757, 8761, 8762, 11925, 11928, 12568, 12569, 12570) 
                            UNION
                            ALL SELECT
                                7146 
                            UNION
                            ALL SELECT
                                7147 
                            UNION
                            ALL SELECT
                                7148 
                            UNION
                            ALL SELECT
                                7149 
                            UNION
                            ALL SELECT
                                8757 
                            UNION
                            ALL SELECT
                                8761 
                            UNION
                            ALL SELECT
                                8762 
                            UNION
                            ALL SELECT
                                11925 
                            UNION
                            ALL SELECT
                                11928 
                            UNION
                            ALL SELECT
                                12568 
                            UNION
                            ALL SELECT
                                12569 
                            UNION
                            ALL SELECT
                                12570))))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
procedureOccurrence_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports",
  "procedureOccurrence",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "procedureOccurrence",
  "procedureOccurrence_*.csv")
message(str_glue('The data will be written to {procedureOccurrence_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", procedureOccurrence_sql, billing = "wb-lucent-greens-2488"),
  procedureOccurrence_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {procedureOccurrence_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

procedureOccurrence_df <- read_bq_export_from_workspace_bucket(procedureOccurrence_path)
dim(procedureOccurrence_df)
head(procedureOccurrence_df, 5)


# Domain ingredientOccurrence
ingredientOccurrence_sql = r"(

    SELECT
        st.days_supply,
        st.dose_unit_source_value,
        st.drug_concept_id,
        st.drug_exposure_end_datetime,
        st.drug_exposure_start_datetime,
        st.drug_source_concept_id,
        st.drug_source_value,
        st.drug_type_concept_id,
        dt0.concept_name AS T_DISP_drug_type_concept_name,
        st.lot_number,
        st.person_id,
        st.quantity,
        st.refills,
        st.route_concept_id,
        dt1.concept_name AS T_DISP_route_concept_name,
        st.route_source_value,
        st.sig,
        dt2.concept_code AS T_DISP_source_concept_code,
        dt2.concept_name AS T_DISP_source_concept_name,
        dt2.vocabulary_id AS T_DISP_source_vocabulary,
        dt3.concept_code AS T_DISP_standard_concept_code,
        dt3.concept_name AS T_DISP_standard_concept_name,
        dt3.vocabulary_id AS T_DISP_standard_vocabulary,
        st.stop_reason,
        st.verbatim_end_date,
        st.visit_occurrence_id 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.drug_exposure AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.drug_type_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.route_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.drug_source_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt3 
            ON dt3.concept_id = st.drug_concept_id 
    WHERE
        st.drug_exposure_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_ingredientOccurrence_person_id 
        WHERE
            (((person_id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
            WHERE
                condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor = 313504 
                UNION
                ALL SELECT
                    313504))) 
                OR ((person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ENG')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ACVRL1')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN ('18-51067120-TAGAC-T', '18-51058355-G-A', '18-51048839-C-T', '18-51076662-C-T', '18-51076653-C-T', '18-51078306-A-G', '18-51078294-C-T', '10-47325545-T-C')))))) 
                AND (drug_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_ingredientConcept_default 
                WHERE
                    ancestor IN (1303425, 1397141, 21601078) 
                UNION
                ALL SELECT
                    1303425 
                UNION
                ALL SELECT
                    1397141 
                UNION
                ALL SELECT
                    21601078))) 
                OR (((person_id IN (SELECT
                    person_id AS primary_id 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
                WHERE
                    condition_concept_id IN (SELECT
                        descendant 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                    WHERE
                        ancestor = 313504 
                    UNION
                    ALL SELECT
                        313504))) 
                    OR ((person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ENG')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                    OR (person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ACVRL1')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                    OR (person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN ('18-51067120-TAGAC-T', '18-51058355-G-A', '18-51048839-C-T', '18-51076662-C-T', '18-51076653-C-T', '18-51078306-A-G', '18-51078294-C-T', '10-47325545-T-C')))))) 
                    AND (id IN (SELECT
                        entity_A_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_RIDS_cpt4Person_ingredientOccurrence_cpt4Concept 
                    WHERE
                        entity_B_id IN (SELECT
                            descendant 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_cpt4Concept_default 
                        WHERE
                            ancestor IN (7146, 7147, 7148, 7149, 8757, 8761, 8762, 11925, 11928, 12568, 12569, 12570) 
                        UNION
                        ALL SELECT
                            7146 
                        UNION
                        ALL SELECT
                            7147 
                        UNION
                        ALL SELECT
                            7148 
                        UNION
                        ALL SELECT
                            7149 
                        UNION
                        ALL SELECT
                            8757 
                        UNION
                        ALL SELECT
                            8761 
                        UNION
                        ALL SELECT
                            8762 
                        UNION
                        ALL SELECT
                            11925 
                        UNION
                        ALL SELECT
                            11928 
                        UNION
                        ALL SELECT
                            12568 
                        UNION
                        ALL SELECT
                            12569 
                        UNION
                        ALL SELECT
                            12570))))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
ingredientOccurrence_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports",
  "ingredientOccurrence",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "ingredientOccurrence",
  "ingredientOccurrence_*.csv")
message(str_glue('The data will be written to {ingredientOccurrence_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", ingredientOccurrence_sql, billing = "wb-lucent-greens-2488"),
  ingredientOccurrence_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {ingredientOccurrence_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

ingredientOccurrence_df <- read_bq_export_from_workspace_bucket(ingredientOccurrence_path)
dim(ingredientOccurrence_df)
head(ingredientOccurrence_df, 5)


# Domain zipcodeSocioeconomic
zipcodeSocioeconomic_sql = r"(

    SELECT
        american_community_survey_year,
        assisted_income,
        deprivation_index,
        high_school_education,
        id,
        median_income,
        no_health_insurance,
        observation_datetime_str,
        observation_datetime_utc,
        person_id,
        poverty,
        vacant_housing,
        zip_code 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_zipcodeSocioeconomic 
    WHERE
        (
            person_id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
            WHERE
                condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor = 313504 
                UNION
                ALL SELECT
                    313504))
            ) 
            OR (
                (
                    person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ENG')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))
                ) 
                OR (
                    person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ACVRL1')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))
                ) 
                OR (
                    person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN ('18-51067120-TAGAC-T', '18-51058355-G-A', '18-51048839-C-T', '18-51076662-C-T', '18-51076653-C-T', '18-51078306-A-G', '18-51078294-C-T', '10-47325545-T-C')))
                )
            ))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
zipcodeSocioeconomic_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports",
  "zipcodeSocioeconomic",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "zipcodeSocioeconomic",
  "zipcodeSocioeconomic_*.csv")
message(str_glue('The data will be written to {zipcodeSocioeconomic_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", zipcodeSocioeconomic_sql, billing = "wb-lucent-greens-2488"),
  zipcodeSocioeconomic_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {zipcodeSocioeconomic_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

zipcodeSocioeconomic_df <- read_bq_export_from_workspace_bucket(zipcodeSocioeconomic_path)
dim(zipcodeSocioeconomic_df)
head(zipcodeSocioeconomic_df, 5)


# Domain deviceOccurrence
deviceOccurrence_sql = r"(

    SELECT
        st.device_concept_id,
        st.device_exposure_end_datetime,
        st.device_exposure_start_datetime,
        st.device_source_concept_id,
        st.device_source_value,
        st.device_type_concept_id,
        dt0.concept_name AS T_DISP_device_type_concept_name,
        st.person_id,
        dt1.concept_code AS T_DISP_source_concept_code,
        dt1.concept_name AS T_DISP_source_concept_name,
        dt1.vocabulary_id AS T_DISP_source_vocabulary,
        dt2.concept_code AS T_DISP_standard_concept_code,
        dt2.concept_name AS T_DISP_standard_concept_name,
        dt2.vocabulary_id AS T_DISP_standard_vocabulary,
        st.visit_occurrence_id 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.device_exposure AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.device_type_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.device_source_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.device_concept_id 
    WHERE
        st.device_exposure_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_deviceOccurrence 
        WHERE
            ((person_id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
            WHERE
                condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor = 313504 
                UNION
                ALL SELECT
                    313504))) 
                OR ((person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ENG')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ACVRL1')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (person_id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN ('18-51067120-TAGAC-T', '18-51058355-G-A', '18-51048839-C-T', '18-51076662-C-T', '18-51076653-C-T', '18-51078306-A-G', '18-51078294-C-T', '10-47325545-T-C')))))) 
                AND (id IN (SELECT
                    entity_A_id 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_RIDS_cpt4Person_deviceOccurrence_cpt4Concept 
                WHERE
                    entity_B_id IN (SELECT
                        descendant 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_cpt4Concept_default 
                    WHERE
                        ancestor IN (7146, 7147, 7148, 7149, 8757, 8761, 8762, 11925, 11928, 12568, 12569, 12570) 
                    UNION
                    ALL SELECT
                        7146 
                    UNION
                    ALL SELECT
                        7147 
                    UNION
                    ALL SELECT
                        7148 
                    UNION
                    ALL SELECT
                        7149 
                    UNION
                    ALL SELECT
                        8757 
                    UNION
                    ALL SELECT
                        8761 
                    UNION
                    ALL SELECT
                        8762 
                    UNION
                    ALL SELECT
                        11925 
                    UNION
                    ALL SELECT
                        11928 
                    UNION
                    ALL SELECT
                        12568 
                    UNION
                    ALL SELECT
                        12569 
                    UNION
                    ALL SELECT
                        12570)))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
deviceOccurrence_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports",
  "deviceOccurrence",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "deviceOccurrence",
  "deviceOccurrence_*.csv")
message(str_glue('The data will be written to {deviceOccurrence_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", deviceOccurrence_sql, billing = "wb-lucent-greens-2488"),
  deviceOccurrence_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {deviceOccurrence_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

deviceOccurrence_df <- read_bq_export_from_workspace_bucket(deviceOccurrence_path)
dim(deviceOccurrence_df)
head(deviceOccurrence_df, 5)


Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done

